In [0]:
%sql

USE CATALOG insurance;
CREATE SCHEMA IF NOT EXISTS bronze;


In [0]:
%python

from src.common import config

# the place the data is retrieved
src = config.volume_path("customers")
# the path for the auto loader check_point
chk = config.checkpoint_path("bronze_customers")

# reading data using spark streaming
(spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv") # the source files are CSV
    .option("header", "true") # first line from the CSV has the column names
    .option("cloudFiles.schemaLocation", chk) # where to keep the schema (like the column name strucutre from above)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # in case a new column appears for example, the auto loader can extend the schema
    .option("rescuedDataColumn", "_rescued_data") # if the data is not in the correct format, the data will be put in _rescued_data
    .load(src) # read the files from the raw volume from the src path
    # first param - keeps all the columns from the csv
    # second param - takes the file path where the raw came from and puts it in the _source_file column 
    # third param - adds the current date and time in the _ingest_ts column
    .selectExpr("*", "_metadata.file_path AS _source_file", "current_timestamp() AS _ingest_ts") 
    .writeStream # start configuring a straming write
    .option("checkpointLocation", chk) # start  the registration of the checkpoint for not processing processed files
    .trigger(availableNow=True) # process the new available files then stop
    .toTable(config.table("bronze", "customers"))) # write the result in a table in the catalog 

In [0]:
%sql
USE CATALOG insurance;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.payments;

-- the command below was AI suggested
ALTER TABLE bronze.payments
SET TBLPROPERTIES (
  'delta.feature.timestampNtz' = 'supported'
);

COPY INTO bronze.payments
FROM '/Volumes/insurance/landing/raw/payments'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true'); -- re-running skips already loaded files